# CLIP ViT-B/16 - Selective INT8 / FP16 TensorRT Optimization

**Reproducible research notebook.** Quantize a pretrained CLIP vision encoder, preserve ONNX quantization scales and INT8 weights, compile mixed-precision and FP16 TensorRT engines, and evaluate identical balanced validation samples. The final **C++ inference application lives in `main.cpp`**, not in this notebook.

**Hardware validated:** NVIDIA Tesla T4, TensorRT 11.4. **Dataset:** `dronefreak/BDD100K-Period-Classification`, pinned revision. **Classes:** daytime, night, dawn/dusk. No model fine-tuning.

> Run cells in order in a GPU-enabled Google Colab environment. Results are written under `CLIP_Quantization_Project/` in Google Drive. Full pipeline needs dataset access, a GPU, and may require a Colab runtime restart after dependency installation. Engines are not portable across every GPU / TensorRT version.

## 1. Setup and deterministic configuration

In [ ]:
# Colab environment: install versions used for this project.
%pip install -q onnx==1.21.0 onnxruntime-gpu==1.24.4 tensorrt==11.4.0.106 \
    nvidia-modelopt==0.47.0 onnxconverter-common transformers accelerate \
    datasets huggingface-hub pandas scikit-learn pillow safetensors tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, copy, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import tensorrt as trt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/CLIP_Quantization_Project")
output_dir = PROJECT_ROOT / "phase2_artifacts"
cache_dir = PROJECT_ROOT / "cache"
data_dir = PROJECT_ROOT / "data"
results_dir = PROJECT_ROOT / "results"
for folder in (output_dir, cache_dir, data_dir, results_dir):
    folder.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache_dir / "huggingface")
os.environ["HF_HUB_CACHE"] = str(cache_dir / "huggingface" / "hub")

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab."
device = torch.device("cuda")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATASET_ID = "dronefreak/BDD100K-Period-Classification"
DATASET_REVISION = "47b8bde4a2d808df9208a89cdd10f68b251bcdb4"
MODEL_ID = "openai/clip-vit-base-patch16"
LABELS = ["daytime", "night", "dawn/dusk"]
PROMPTS = [
    "A photograph of a road during daytime",
    "A photograph of a road at night",
    "A photograph of a road at dawn or dusk",
]
print("GPU:", torch.cuda.get_device_name(0), "| TensorRT:", trt.__version__)

## 2. Load the frozen CLIP model and generate text embeddings

In [ ]:
clip_model = CLIPModel.from_pretrained(MODEL_ID, dtype=torch.float32).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)

class CLIPVisionEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision_model = model.vision_model
        self.visual_projection = model.visual_projection
    def forward(self, pixel_values):
        outputs = self.vision_model(pixel_values=pixel_values)
        return self.visual_projection(outputs.pooler_output)

vision_encoder = CLIPVisionEncoder(clip_model).to(device).eval()
text_inputs = clip_processor(text=PROMPTS, return_tensors="pt", padding=True)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
with torch.inference_mode():
    text_features = clip_model.get_text_features(**text_inputs)
    if not isinstance(text_features, torch.Tensor):
        text_features = text_features.pooler_output
    text_features = F.normalize(text_features.float(), dim=-1)
assert text_features.shape == (3, 512)

# For standalone C++ classifier
cpp_dir = PROJECT_ROOT / "cpp_deployment"
cpp_dir.mkdir(parents=True, exist_ok=True)
text_features.detach().cpu().numpy().astype("<f4").tofile(cpp_dir / "text_embeddings_fp32.bin")
(cpp_dir / "labels.json").write_text(json.dumps({
    "labels": LABELS, "prompts": PROMPTS,
    "embedding_dimension": 512, "text_embedding_layout": "row_major_float32"
}, indent=2))
print("C++ text embedding file:", cpp_dir / "text_embeddings_fp32.bin")

## 3. Reconstruct balanced calibration and evaluation samples

300 training images for calibration and 300 separate validation images for evaluation (100 images per class). The metadata revision, ordering, and seeds are fixed. **Downloaded dataset images are excluded from Git.**

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

files = HfApi().list_repo_files(repo_id=DATASET_ID, repo_type="dataset", revision=DATASET_REVISION)
metadata_files = sorted(f for f in files if f.endswith("/metadata.jsonl"))
assert metadata_files
metadata_frames = []
for repo_path in metadata_files:
    local = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename=repo_path,
                            revision=DATASET_REVISION, local_dir=str(data_dir / "metadata"))
    frame = pd.read_json(local, lines=True)
    frame["source_metadata"] = repo_path
    frame["source_split"] = "validation" if "/val/" in repo_path else "train"
    metadata_frames.append(frame)
all_metadata = pd.concat(metadata_frames, ignore_index=True)
print("Metadata records:", len(all_metadata))

In [ ]:
import pandas as pd
from pathlib import PurePosixPath

SEED = 42
SAMPLES_PER_CLASS = 100

class_to_index = {
    label: i for i, label in enumerate(LABELS)
}

def create_balanced_manifest(metadata, split, seed):
    subset = metadata[
        (metadata["source_split"] == split)
        & (metadata["label"].isin(LABELS))
    ].copy()

    selected = (
        subset.groupby("label", group_keys=False)
        .sample(
            n=SAMPLES_PER_CLASS,
            random_state=seed
        )
        .sample(
            frac=1,
            random_state=seed
        )
        .reset_index(drop=True)
    )

    # Metadata lives in the same shard as each image.
    selected["repo_path"] = selected.apply(
        lambda row: str(
            PurePosixPath(row["source_metadata"]).parent
            / row["file_name"]
        ),
        axis=1
    )

    selected["label_index"] = (
        selected["label"].map(class_to_index)
    )

    selected["local_path"] = selected["repo_path"].map(
        lambda p: str(data_dir / p)
    )

    return selected


df_calibration_manifest = create_balanced_manifest(
    all_metadata,
    split="train",
    seed=SEED
)

df_eval_manifest = create_balanced_manifest(
    all_metadata,
    split="validation",
    seed=SEED + 1
)

# Check that the two sets are disjoint.
overlap = set(
    df_calibration_manifest["repo_path"]
) & set(
    df_eval_manifest["repo_path"]
)

assert len(overlap) == 0

# Persist both manifests immediately.
calibration_manifest_path = (
    data_dir / "calibration_manifest.csv"
)

evaluation_manifest_path = (
    data_dir / "evaluation_manifest.csv"
)

df_calibration_manifest.to_csv(
    calibration_manifest_path,
    index=False
)

df_eval_manifest.to_csv(
    evaluation_manifest_path,
    index=False
)

print("Calibration:", df_calibration_manifest["label"].value_counts().to_dict())
print("Evaluation:", df_eval_manifest["label"].value_counts().to_dict())

In [ ]:
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
import shutil

combined_manifest = pd.concat(
    [
        df_calibration_manifest,
        df_eval_manifest
    ],
    ignore_index=True
)

download_jobs = (
    combined_manifest[
        ["repo_path", "local_path"]
    ]
    .drop_duplicates()
    .to_dict("records")
)

def download_image(job):
    destination = Path(job["local_path"])

    if destination.exists() and destination.stat().st_size > 0:
        return "cached", job["repo_path"]

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    source = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename=job["repo_path"],
        revision=DATASET_REVISION,
        cache_dir=str(cache_dir / "huggingface_downloads")
    )

    shutil.copy2(source, destination)

    return "downloaded", job["repo_path"]


download_counts = {
    "cached": 0,
    "downloaded": 0,
    "failed": 0
}

failures = []

# A small worker pool avoids excessive Hugging Face requests
# and simultaneous Google Drive writes.
with ThreadPoolExecutor(max_workers=4) as executor:

    futures = {
        executor.submit(download_image, job): job
        for job in download_jobs
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Restoring images"
    ):
        try:
            status, repo_path = future.result()
            download_counts[status] += 1

        except Exception as exc:
            download_counts["failed"] += 1
            failures.append(
                (futures[future]["repo_path"], str(exc))
            )

print("Download summary:", download_counts)
assert not failures, f"Download errors: {failures[:3]}"

In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class CLIPCalibrationDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.iloc[index]["local_path"]

        with Image.open(image_path) as image:
            pixel_values = self.processor(
                images=image.convert("RGB"),
                return_tensors="pt"
            )["pixel_values"].squeeze(0)

        return pixel_values


# Confirm all image files exist.
for name, manifest in [
    ("Calibration", df_calibration_manifest),
    ("Evaluation", df_eval_manifest)
]:
    missing = [
        p for p in manifest["local_path"]
        if not Path(p).is_file()
    ]

    assert not missing, (
        f"{name} set has {len(missing)} missing images"
    )

calibration_dataset = CLIPCalibrationDataset(
    df_calibration_manifest,
    clip_processor
)

evaluation_dataset = CLIPCalibrationDataset(
    df_eval_manifest,
    clip_processor
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Match LABELS ordering exactly.
y_true = df_eval_manifest["label"].map({
    label: i for i, label in enumerate(LABELS)
}).to_numpy(dtype=np.int64)

assert len(calibration_dataset) == 300
assert len(evaluation_dataset) == 300
assert not np.any(pd.isna(y_true))

assert next(iter(calibration_loader)).shape == (8, 3, 224, 224)
print("Samples:", len(calibration_dataset), "calibration,", len(evaluation_dataset), "evaluation")

## 4. Establish the FP32 reference

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import json

vision_encoder.eval()

fp32_predictions = []
fp32_embeddings = []

with torch.inference_mode():
    for images in evaluation_loader:
        images = images.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True
        )

        features = vision_encoder(images)

        features = F.normalize(
            features.float(),
            dim=-1
        )

        normalized_text = F.normalize(
            text_features.float(),
            dim=-1
        )

        scores = features @ normalized_text.T

        fp32_predictions.extend(
            scores.argmax(dim=-1).cpu().numpy().tolist()
        )

        fp32_embeddings.append(features.cpu())

fp32_predictions = np.asarray(
    fp32_predictions,
    dtype=np.int64
)

fp32_embeddings = torch.cat(fp32_embeddings, dim=0)

fp32_accuracy = accuracy_score(y_true, fp32_predictions)
fp32_macro_f1 = f1_score(
    y_true,
    fp32_predictions,
    labels=[0, 1, 2],
    average="macro"
)

print("Recovered FP32 baseline")
print("=" * 40)
print("Images:", len(fp32_predictions))
print(f"Accuracy: {fp32_accuracy:.4f}")
print(f"Macro F1: {fp32_macro_f1:.4f}")

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        fp32_predictions,
        labels=[0, 1, 2]
    )
)

# Save evaluation predictions to Google Drive.
np.save(
    results_dir / "fp32_predictions.npy",
    fp32_predictions
)

np.save(
    results_dir / "fp32_embeddings.npy",
    fp32_embeddings.numpy()
)

baseline_results = {
    "model": MODEL_ID,
    "dataset": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "evaluation_images": len(y_true),
    "accuracy": float(fp32_accuracy),
    "macro_f1": float(fp32_macro_f1),
    "labels": LABELS,
    "seed": SEED + 1
}

(results_dir / "fp32_baseline.json").write_text(
    json.dumps(baseline_results, indent=2)
)

print("\nFP32 predictions, embeddings, and metrics saved to Drive.")

## 5. Selective INT8 calibration and checkpoint

Quantize attention Q/K/V/output projections, MLP `fc1` and selected embedding quantizers. Preserve `fc2` and normalization precision. Later, check the **actual enabled quantizers**, not just the configuration strings.

In [ ]:
import copy
import time
import torch
import modelopt.torch.quantization as mtq

# Explicitly enable only the selected quantization regions.
# Start with all quantizers disabled.
selective_int8_cfg = {
    "quant_cfg": [
        {"quantizer_name": "*", "enable": False},

        # Attention Q, K, V and output projections.
        {
            "quantizer_name": "*self_attn.*_proj.weight_quantizer",
            "cfg": {"num_bits": 8, "axis": 0}
        },
        {
            "quantizer_name": "*self_attn.*_proj.input_quantizer",
            "cfg": {"num_bits": 8, "axis": None}
        },

        # MLP fc1 only; fc2 remains unquantized.
        {
            "quantizer_name": "*mlp.fc1.weight_quantizer",
            "cfg": {"num_bits": 8, "axis": 0}
        },
        {
            "quantizer_name": "*mlp.fc1.input_quantizer",
            "cfg": {"num_bits": 8, "axis": None}
        },

        # Embedding quantizers from our earlier configuration.
        {
            "quantizer_name": "*patch_embedding.weight_quantizer",
            "cfg": {"num_bits": 8, "axis": 0}
        },
        {
            "quantizer_name": "*patch_embedding.input_quantizer",
            "cfg": {"num_bits": 8, "axis": None}
        },
        {
            "quantizer_name": "*position_embedding.weight_quantizer",
            "cfg": {"num_bits": 8, "axis": 0}
        },
        {
            "quantizer_name": "*position_embedding.input_quantizer",
            "cfg": {"num_bits": 8, "axis": None}
        },
    ],
    "algorithm": "max"
}

# Preserve the original FP32 model.
quantized_encoder = copy.deepcopy(
    vision_encoder
).to(device).eval()

def calibration_forward_loop(model):
    model.eval()
    batches = 0
    images_seen = 0

    with torch.inference_mode():
        for images in calibration_loader:
            images = images.to(
                device=device,
                dtype=torch.float32,
                non_blocking=True
            )

            model(images)

            batches += 1
            images_seen += images.shape[0]

    print("Calibration batches:", batches)
    assert next(iter(calibration_loader)).shape == (8, 3, 224, 224)
print("Samples:", len(calibration_dataset), "calibration,", len(evaluation_dataset), "evaluation")

In [ ]:
import pandas as pd

quantizer_rows = []

for name, module in quantized_encoder.named_modules():
    if name.endswith((
        "input_quantizer",
        "weight_quantizer"
    )):
        enabled = bool(getattr(module, "is_enabled", False))

        quantizer_rows.append({
            "name": name,
            "enabled": enabled,
            "region": (
                "attention"
                if ".self_attn." in name
                else "fc1"
                if ".mlp.fc1." in name
                else "fc2"
                if ".mlp.fc2." in name
                else "embedding"
                if "embedding" in name
                else "other"
            )
        })

quantizer_inventory = pd.DataFrame(quantizer_rows)
print("Enabled quantizers:", int(quantizer_inventory["enabled"].sum()))
print("Enabled by region:", quantizer_inventory[quantizer_inventory["enabled"]].groupby("region").size().to_dict())
assert not quantizer_inventory[quantizer_inventory["enabled"] & quantizer_inventory["region"].isin(["fc2", "other"])].shape[0]


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

quantized_encoder.eval()

int8_predictions = []
int8_embeddings = []

with torch.inference_mode():
    for images in evaluation_loader:
        images = images.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True
        )

        features = quantized_encoder(images)

        features = F.normalize(
            features.float(),
            dim=-1
        )

        scores = (
            features
            @ F.normalize(text_features.float(), dim=-1).T
        )

        int8_predictions.extend(
            scores.argmax(dim=-1).cpu().numpy()
        )

        int8_embeddings.append(features.cpu())

int8_predictions = np.asarray(
    int8_predictions,
    dtype=np.int64
)

int8_embeddings = torch.cat(
    int8_embeddings,
    dim=0
)

int8_accuracy = accuracy_score(
    y_true,
    int8_predictions
)

int8_macro_f1 = f1_score(
    y_true,
    int8_predictions,
    labels=[0, 1, 2],
    average="macro"
)

int8_agreement = np.mean(
    int8_predictions == fp32_predictions
)

embedding_cosine = F.cosine_similarity(
    int8_embeddings,
    fp32_embeddings,
    dim=-1
)

print("Selective INT8 PyTorch evaluation")
print("=" * 45)
print("Images evaluated:", len(int8_predictions))
print(f"Accuracy: {int8_accuracy:.4f}")
print(f"Macro F1: {int8_macro_f1:.4f}")
print(f"FP32 agreement: {int8_agreement:.4f}")
print("Mean embedding cosine:", f"{embedding_cosine.mean().item():.6f}")

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        int8_predictions,
        labels=[0, 1, 2]
    )
)

print("\nFP32 accuracy:", f"{fp32_accuracy:.4f}")
print(
    "Accuracy difference:",
    f"{(int8_accuracy - fp32_accuracy) * 100:+.2f} pp"
)

In [ ]:
import json
import modelopt.torch.opt as mto

checkpoint_path = (
    output_dir / "clip_selective_int8_modelopt.pth"
)

print("Saving calibrated ModelOpt checkpoint...")

mto.save(
    quantized_encoder,
    str(checkpoint_path)
)

assert checkpoint_path.is_file()

print("Checkpoint saved:", checkpoint_path)
print(
    "Checkpoint size:",
    f"{checkpoint_path.stat().st_size / 1024**2:.2f} MiB"
)

# Save metrics and quantizer inventory separately.
np.save(
    results_dir / "selective_int8_predictions.npy",
    int8_predictions
)

quantizer_inventory.to_csv(
    results_dir / "selective_int8_quantizers.csv",
    index=False
)

metrics = {
    "model": MODEL_ID,
    "dataset": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "evaluation_images": len(y_true),
    "enabled_quantizers": int(
        quantizer_inventory["enabled"].sum()
    ),
    "accuracy": float(int8_accuracy),
    "macro_f1": float(int8_macro_f1),
    "fp32_agreement": float(int8_agreement),
    "mean_embedding_cosine": float(
        embedding_cosine.mean().item()
    )
}

(
    results_dir / "selective_int8_metrics.json"
).write_text(
    json.dumps(metrics, indent=2)
)

print("\nPredictions, quantizer inventory, and metrics saved.")
print("All checkpoint files are stored on Google Drive.")

## 6. Export selective INT8 ONNX and verify its Q/DQ structure

In [ ]:
import torch
import time

int8_onnx_path = (
    output_dir / "clip_attention_fc1_int8_qdq.onnx"
)

quantized_encoder.eval()

dummy_input = torch.randn(
    1, 3, 224, 224,
    device=device,
    dtype=torch.float32
)

print("Exporting selective INT8 model to ONNX...")

start = time.perf_counter()

with torch.inference_mode():
    torch.onnx.export(
        quantized_encoder,
        (dummy_input,),
        str(int8_onnx_path),
        input_names=["pixel_values"],
        output_names=["image_features"],
        dynamic_axes={
            "pixel_values": {0: "batch_size"},
            "image_features": {0: "batch_size"}
        },
        opset_version=17,
        do_constant_folding=True,
        dynamo=False
    )

export_seconds = time.perf_counter() - start

assert int8_onnx_path.exists()

print("\nONNX export completed!")
print(f"Export time: {export_seconds:.2f} s")
print("Saved to:", int8_onnx_path)
print(
    "File size:",
    f"{int8_onnx_path.stat().st_size / 1024**2:.2f} MiB"
)

In [ ]:
import onnx
from collections import Counter

qdq_model = onnx.load(str(int8_onnx_path))

print("Validating ONNX model...")
onnx.checker.check_model(qdq_model)
print("ONNX checker: PASSED")

op_counts = Counter(
    node.op_type
    for node in qdq_model.graph.node
)

print("\nTotal ONNX nodes:", len(qdq_model.graph.node))

for op in [
    "QuantizeLinear",
    "DequantizeLinear",
    "MatMul",
    "LayerNormalization",
    "Softmax",
    "Conv",
    "Cast"
]:
    print(f"{op:22s}: {op_counts.get(op, 0)}")

q_count = op_counts["QuantizeLinear"]
dq_count = op_counts["DequantizeLinear"]
assert q_count == dq_count and q_count >= 120
print("Valid Q/DQ pairs:", q_count)

## 7. Convert to mixed INT8/FP16 and preserve quantized values

Restore all original FP32 Q/DQ scales after conversion, then verify all quantized weight integers remain unchanged. These assertions catch silent accuracy regressions.

In [ ]:
import copy
import time
import numpy as np
import onnx

from onnx import numpy_helper
from onnxconverter_common import float16
from collections import Counter

# Resolve initializer and Constant values, including Cast chains.
def make_constant_resolver(model):
    values = {
        t.name: numpy_helper.to_array(t)
        for t in model.graph.initializer
    }

    producers = {
        output: node
        for node in model.graph.node
        for output in node.output
    }

    def resolve(name, visited=None):
        if visited is None:
            visited = set()

        if name in visited:
            return None

        visited.add(name)

        if name in values:
            return values[name]

        node = producers.get(name)

        if node is None:
            return None

        if node.op_type == "Constant":
            for attr in node.attribute:
                if attr.name == "value":
                    return numpy_helper.to_array(attr.t)

        if node.op_type in ("Cast", "Identity"):
            source = resolve(node.input[0], visited)

            if source is None:
                return None

            if node.op_type == "Identity":
                return source

            to_attr = next(
                (a for a in node.attribute if a.name == "to"),
                None
            )

            if to_attr is None:
                return None

            dtype = onnx.helper.tensor_dtype_to_np_dtype(
                to_attr.i
            )

            return source.astype(dtype)

        return None

    return resolve


def extract_qdq_scales(model):
    resolve = make_constant_resolver(model)
    result = {}

    for node in model.graph.node:
        if node.op_type not in (
            "QuantizeLinear",
            "DequantizeLinear"
        ):
            continue

        result[node.name] = resolve(node.input[1])

    return result


# Original calibration scales.
original_qdq_scales = extract_qdq_scales(qdq_model)

assert all(
    value is not None
    for value in original_qdq_scales.values()
)

print(
    "Original Q/DQ scale inputs:",
    len(original_qdq_scales)
)

print("\nConverting floating-point computation to FP16...")

start = time.perf_counter()

mixed_fp16_model = float16.convert_float_to_float16(
    qdq_model,
    keep_io_types=True,
    disable_shape_infer=False,
    op_block_list=[
        "QuantizeLinear",
        "DequantizeLinear"
    ]
)

print("FP16 conversion completed.")

# Restore original FP32 calibration scales.
repaired_model = copy.deepcopy(mixed_fp16_model)

restored_count = 0

for node in repaired_model.graph.node:

    if node.op_type not in (
        "QuantizeLinear",
        "DequantizeLinear"
    ):
        continue

    scale = original_qdq_scales[node.name]

    if scale is None:
        raise RuntimeError(
            f"Missing original scale: {node.name}"
        )

    scale_name = (
        f"restored_qdq_scale_{restored_count}"
    )

    repaired_model.graph.initializer.append(
        numpy_helper.from_array(
            np.asarray(scale, dtype=np.float32),
            name=scale_name
        )
    )

    node.input[1] = scale_name

    restored_count += 1


mixed_onnx_path = (
    output_dir /
    "clip_attention_fc1_int8_fp16_repaired.onnx"
)

onnx.save(
    repaired_model,
    str(mixed_onnx_path)
)

conversion_seconds = time.perf_counter() - start

print("\nMixed ONNX model saved!")
print("Path:", mixed_onnx_path)
print(
    "Size:",
    f"{mixed_onnx_path.stat().st_size / 1024**2:.2f} MiB"
)
print("Restored scales:", restored_count)
print(f"Conversion time: {conversion_seconds:.2f} s")

print("\nOperation counts:")
counts = Counter(
    node.op_type
    for node in repaired_model.graph.node
)

for op in [
    "QuantizeLinear",
    "DequantizeLinear",
    "MatMul",
    "LayerNormalization",
    "Cast"
]:
    print(f"{op}: {counts[op]}")

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto

print("Validating mixed INT8/FP16 ONNX...")

onnx.checker.check_model(
    onnx.load(str(mixed_onnx_path))
)

print("ONNX checker: PASSED")

repaired_scales = extract_qdq_scales(
    repaired_model
)

scale_mismatches = [
    name
    for name in original_qdq_scales
    if (
        name not in repaired_scales
        or repaired_scales[name] is None
        or not np.array_equal(
            original_qdq_scales[name],
            repaired_scales[name]
        )
    )
]

print("\nQ/DQ scales checked:", len(original_qdq_scales))
print("Scale mismatches:", len(scale_mismatches))

assert not scale_mismatches

initializers = {
    t.name: t
    for t in repaired_model.graph.initializer
}

fc2_weight_types = []

for node in repaired_model.graph.node:
    if (
        node.op_type == "MatMul"
        and "/mlp/fc2/" in node.name
    ):
        for name in node.input:
            if name in initializers:
                fc2_weight_types.append(
                    TensorProto.DataType.Name(
                        initializers[name].data_type
                    )
                )

print("\nfc2 weights found:", len(fc2_weight_types))
print("fc2 weight types:", set(fc2_weight_types))

assert len(fc2_weight_types) == 12
assert set(fc2_weight_types) == {"FLOAT16"}

print("\nModel I/O:")
for item in repaired_model.graph.input:
    print(
        "Input:",
        item.name,
        TensorProto.DataType.Name(
            item.type.tensor_type.elem_type
        )
    )

for item in repaired_model.graph.output:
    print(
        "Output:",
        item.name,
        TensorProto.DataType.Name(
            item.type.tensor_type.elem_type
        )
    )

print("\nMixed-precision ONNX validation successful!")
print("Artifact saved permanently to Google Drive.")

In [ ]:
import numpy as np
import onnx

original_resolve = make_constant_resolver(qdq_model)
repaired_resolve = make_constant_resolver(repaired_model)

original_nodes = {
    node.name: node
    for node in qdq_model.graph.node
}

repaired_nodes = {
    node.name: node
    for node in repaired_model.graph.node
}

def get_attribute(node, name, default=None):
    for attr in node.attribute:
        if attr.name == name:
            return onnx.helper.get_attribute_value(attr)
    return default

def broadcast_parameter(value, shape, axis):
    value = np.asarray(value)

    if value.size == 1:
        return value.reshape(())

    axis = axis % len(shape)

    if value.ndim != 1 or value.size != shape[axis]:
        raise ValueError(
            f"Invalid quantization parameter shape: "
            f"{value.shape}, weights={shape}, axis={axis}"
        )

    new_shape = [1] * len(shape)
    new_shape[axis] = value.size

    return value.reshape(new_shape)

def quantized_values(node, resolver):
    weights = resolver(node.input[0])
    scale = resolver(node.input[1])

    if weights is None or scale is None:
        return None

    axis = get_attribute(node, "axis", 1)

    scale = broadcast_parameter(
        scale, weights.shape, axis
    ).astype(np.float64)

    if len(node.input) >= 3 and node.input[2]:
        zp = resolver(node.input[2])

        if zp is None:
            return None

        qdtype = zp.dtype
        zero_point = broadcast_parameter(
            zp, weights.shape, axis
        ).astype(np.int64)
    else:
        qdtype = np.dtype("uint8")
        zero_point = 0

    if np.dtype(qdtype) == np.dtype("int8"):
        qmin, qmax = -128, 127
    elif np.dtype(qdtype) == np.dtype("uint8"):
        qmin, qmax = 0, 255
    else:
        raise ValueError(
            f"Unsupported quantized dtype: {qdtype}"
        )

    return np.clip(
        np.rint(weights.astype(np.float64) / scale)
        + zero_point,
        qmin,
        qmax
    ).astype(qdtype)

results = []

for name, node in original_nodes.items():
    if (
        node.op_type != "QuantizeLinear"
        or "weight_quantizer" not in name
    ):
        continue

    new_node = repaired_nodes[name]

    original = quantized_values(
        node, original_resolve
    )

    converted = quantized_values(
        new_node, repaired_resolve
    )

    if original is None or converted is None:
        raise RuntimeError(
            f"Unable to resolve quantized weights: {name}"
        )

    assert original.shape == converted.shape

    differences = np.count_nonzero(
        original != converted
    )

    results.append({
        "node": name,
        "weight_count": original.size,
        "changed": int(differences)
    })

total_weights = sum(
    row["weight_count"] for row in results
)

changed_weights = sum(
    row["changed"] for row in results
)

print("Weight quantizers checked:", len(results))
print("Total INT8 weight values:", total_weights)
print("Changed values:", changed_weights)

print(
    "Changed percentage:",
    f"{100 * changed_weights / total_weights:.8f}%"
)

print("\nQuantizers containing changed weights:")
for row in results:
    if row["changed"]:
        print(
            row["node"],
            row["changed"],
            "/",
            row["weight_count"]
        )

assert changed_weights == 0, (
    "INT8 weight values changed during FP16 conversion."
)

print("\nINT8 weight preservation verified!")

## 8. Compile the mixed INT8/FP16 TensorRT engine

Optimization profile: min batch 1, optimal batch 8, max batch 32. TensorRT 11.4 uses explicit ONNX precision (no obsolete FP16/INT8 builder flags).

In [ ]:
import tensorrt as trt

mixed_logger = trt.Logger(trt.Logger.INFO)

mixed_builder = trt.Builder(mixed_logger)
mixed_network = mixed_builder.create_network()

mixed_parser = trt.OnnxParser(
    mixed_network,
    mixed_logger
)

print("Parsing mixed INT8/FP16 ONNX model...")

with open(mixed_onnx_path, "rb") as f:
    parsed = mixed_parser.parse(f.read())

if not parsed:
    for i in range(mixed_parser.num_errors):
        print(mixed_parser.get_error(i))

    raise RuntimeError(
        "TensorRT ONNX parsing failed."
    )

print("\nTensorRT parsing successful!")
print("Network layers:", mixed_network.num_layers)

print("\nInput tensors:")
for i in range(mixed_network.num_inputs):
    tensor = mixed_network.get_input(i)
    print(tensor.name, tensor.shape, tensor.dtype)

print("\nOutput tensors:")
for i in range(mixed_network.num_outputs):
    tensor = mixed_network.get_output(i)
    print(tensor.name, tensor.shape, tensor.dtype)

In [ ]:
import time
import tensorrt as trt

# Build configuration
mixed_config = mixed_builder.create_builder_config()

mixed_config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    2 << 30
)

mixed_config.profiling_verbosity = (
    trt.ProfilingVerbosity.DETAILED
)

# Dynamic batch support: 1 to 32
mixed_profile = mixed_builder.create_optimization_profile()

mixed_profile.set_shape(
    "pixel_values",
    min=(1, 3, 224, 224),
    opt=(8, 3, 224, 224),
    max=(32, 3, 224, 224)
)

assert bool(mixed_profile)

mixed_config.add_optimization_profile(mixed_profile)

# Persistent Google Drive destination
mixed_engine_path = (
    output_dir /
    "clip_attention_fc1_int8_fp16_repaired.engine"
)

print("Building mixed INT8/FP16 TensorRT engine...")
print("GPU:", torch.cuda.get_device_name(0))

start = time.perf_counter()

mixed_serialized = mixed_builder.build_serialized_network(
    mixed_network,
    mixed_config
)

if mixed_serialized is None:
    raise RuntimeError(
        "TensorRT engine build failed."
    )

build_seconds = time.perf_counter() - start

# Save immediately to Google Drive.
with open(mixed_engine_path, "wb") as f:
    f.write(mixed_serialized)

assert mixed_engine_path.is_file()

# Deserialize the saved engine.
mixed_runtime = trt.Runtime(mixed_logger)

mixed_engine = mixed_runtime.deserialize_cuda_engine(
    mixed_engine_path.read_bytes()
)

if mixed_engine is None:
    raise RuntimeError(
        "TensorRT engine deserialization failed."
    )

print("\nEngine built and saved successfully!")
print(f"Build time: {build_seconds:.2f} seconds")
print(
    "Engine size:",
    f"{mixed_engine_path.stat().st_size / 1024**2:.2f} MiB"
)
print("Saved to:", mixed_engine_path)

print("\nEngine input/output:")
for i in range(mixed_engine.num_io_tensors):
    name = mixed_engine.get_tensor_name(i)

    print(
        name,
        mixed_engine.get_tensor_dtype(name),
        mixed_engine.get_tensor_shape(name)
    )

## 9. Evaluate mixed engine accuracy

In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

mixed_context = mixed_engine.create_execution_context()
assert mixed_context is not None

mixed_predictions = []
mixed_embeddings = []

input_name = "pixel_values"
output_name = "image_features"

stream = torch.cuda.current_stream()



print("Evaluating rebuilt TensorRT engine...")

with torch.inference_mode():

    normalized_text = F.normalize(
        text_features.float().to(device),
        dim=-1
    )

    for images in evaluation_loader:

        images = images.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True
        ).contiguous()

        mixed_context.set_input_shape(
            input_name,
            tuple(images.shape)
        )

        output_shape = tuple(
            mixed_context.get_tensor_shape(output_name)
        )

        features = torch.empty(
            output_shape,
            device=device,
            dtype=torch.float32
        )

        mixed_context.set_tensor_address(
            input_name,
            images.data_ptr()
        )

        mixed_context.set_tensor_address(
            output_name,
            features.data_ptr()
        )

        ok = mixed_context.execute_async_v3(
            stream_handle=stream.cuda_stream
        )

        if not ok:
            raise RuntimeError("TensorRT execution failed")

        normalized_features = F.normalize(
            features.float(),
            dim=-1
        )

        scores = normalized_features @ normalized_text.T

        mixed_predictions.extend(
            scores.argmax(dim=-1).cpu().numpy().tolist()
        )

        mixed_embeddings.append(
            normalized_features.cpu()
        )

stream.synchronize()

mixed_predictions = np.asarray(
    mixed_predictions,
    dtype=np.int64
)

mixed_embeddings = torch.cat(
    mixed_embeddings,
    dim=0
)

assert len(mixed_predictions) == len(y_true) == 300

mixed_accuracy = accuracy_score(
    y_true, mixed_predictions
)

mixed_macro_f1 = f1_score(
    y_true,
    mixed_predictions,
    labels=[0, 1, 2],
    average="macro"
)

mixed_agreement = np.mean(
    mixed_predictions == fp32_predictions
)

embedding_cosine = F.cosine_similarity(
    mixed_embeddings,
    fp32_embeddings,
    dim=-1
)

print("\n" + "=" * 48)
print("FINAL TENSORRT MIXED INT8/FP16 EVALUATION")
print("=" * 48)

print(f"Images evaluated: {len(y_true)}")
print(f"FP32 accuracy: {fp32_accuracy:.4f}")
print(f"Mixed accuracy: {mixed_accuracy:.4f}")
print(f"Mixed macro F1: {mixed_macro_f1:.4f}")
print(f"FP32 agreement: {mixed_agreement:.4f}")
print(
    f"Mean embedding cosine: "
    f"{embedding_cosine.mean().item():.6f}"
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        mixed_predictions,
        labels=[0, 1, 2]
    )
)

print("\nPer-class results:")
print(
    classification_report(
        y_true,
        mixed_predictions,
        labels=[0, 1, 2],
        target_names=LABELS,
        digits=4
    )
)

# Save all results to persistent Google Drive.
np.save(
    results_dir / "mixed_trt_predictions.npy",
    mixed_predictions
)

np.save(
    results_dir / "mixed_trt_embeddings.npy",
    mixed_embeddings.numpy()
)

metrics = {
    "model": MODEL_ID,
    "precision": "selective_INT8_FP16",
    "dataset": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "evaluation_images": 300,
    "accuracy": float(mixed_accuracy),
    "macro_f1": float(mixed_macro_f1),
    "fp32_agreement": float(mixed_agreement),
    "mean_embedding_cosine": float(
        embedding_cosine.mean().item()
    ),
    "engine_size_mib": float(
        mixed_engine_path.stat().st_size / 1024**2
    ),
    "gpu": torch.cuda.get_device_name(0)
}

(
    results_dir / "mixed_trt_metrics.json"
).write_text(
    json.dumps(metrics, indent=2)
)

print("\nResults saved to Google Drive.")

## 10. Build the FP16-only TensorRT comparison engine

Keep FP32 model I/O while using FP16 weights. Insert an explicit Cast into the patch-embedding convolution to satisfy TensorRT typed inputs.

In [ ]:
import torch
import onnx
from onnxconverter_common import float16
from collections import Counter

fp32_onnx_path = output_dir / "clip_baseline_fp32.onnx"
fp16_onnx_path = output_dir / "clip_baseline_fp16.onnx"

vision_encoder.eval()

dummy_input = torch.randn(
    1, 3, 224, 224,
    device=device,
    dtype=torch.float32
)

print("Exporting original FP32 CLIP vision encoder...")

with torch.inference_mode():
    torch.onnx.export(
        vision_encoder,
        (dummy_input,),
        str(fp32_onnx_path),
        input_names=["pixel_values"],
        output_names=["image_features"],
        dynamic_axes={
            "pixel_values": {0: "batch_size"},
            "image_features": {0: "batch_size"}
        },
        opset_version=17,
        do_constant_folding=True,
        dynamo=False
    )

fp32_baseline_onnx = onnx.load(str(fp32_onnx_path))
onnx.checker.check_model(fp32_baseline_onnx)

print("Converting baseline ONNX to FP16...")

fp16_baseline_onnx = float16.convert_float_to_float16(
    fp32_baseline_onnx,
    keep_io_types=True,
    disable_shape_infer=False
)

onnx.checker.check_model(fp16_baseline_onnx)
onnx.save(fp16_baseline_onnx, str(fp16_onnx_path))

counts = Counter(
    n.op_type for n in fp16_baseline_onnx.graph.node
)

print("\nFP16 baseline ONNX saved:", fp16_onnx_path)
print(f"Size: {fp16_onnx_path.stat().st_size / 1024**2:.2f} MiB")
print("MatMul nodes:", counts["MatMul"])
print("QuantizeLinear nodes:", counts["QuantizeLinear"])
print("DequantizeLinear nodes:", counts["DequantizeLinear"])
print("ONNX validation: PASSED")

In [ ]:
import onnx
import tensorrt as trt
from onnx import helper, TensorProto

# Reload the already-saved FP16 model.
fixed_fp16 = onnx.load(str(fp16_onnx_path))

conv_nodes = [
    node for node in fixed_fp16.graph.node
    if node.op_type == "Conv"
    and "patch_embedding" in node.name
]

assert len(conv_nodes) == 1, (
    f"Expected one patch convolution, found {len(conv_nodes)}"
)

conv_node = conv_nodes[0]

original_input = conv_node.input[0]
cast_output = "patch_embedding_input_fp16"

# Insert a cast directly before the convolution.
cast_node = helper.make_node(
    "Cast",
    inputs=[original_input],
    outputs=[cast_output],
    to=TensorProto.FLOAT16,
    name="patch_embedding_input_cast_fp16"
)

conv_index = next(
    i for i, node in enumerate(fixed_fp16.graph.node)
    if node is conv_node
)

fixed_fp16.graph.node.insert(conv_index, cast_node)
conv_node.input[0] = cast_output

fixed_fp16_path = (
    output_dir / "clip_baseline_fp16_fixed.onnx"
)

onnx.checker.check_model(fixed_fp16)
onnx.save(fixed_fp16, str(fixed_fp16_path))

print("Patched convolution input:")
print("Source:", original_input)
print("Cast output:", conv_node.input[0])
print("Cast target: FLOAT16")
print("\nONNX checker: PASSED")

# Confirm TensorRT accepts the corrected graph.
fp16_logger = trt.Logger(trt.Logger.WARNING)
fp16_builder = trt.Builder(fp16_logger)
fp16_network = fp16_builder.create_network()
fp16_parser = trt.OnnxParser(fp16_network, fp16_logger)

with open(fixed_fp16_path, "rb") as f:
    parsed = fp16_parser.parse(f.read())

if not parsed:
    for i in range(fp16_parser.num_errors):
        print(fp16_parser.get_error(i))
    raise RuntimeError("FP16 TensorRT parsing still failed")

print("\nTensorRT parsing: PASSED")
print("Network layers:", fp16_network.num_layers)
print("Fixed ONNX saved:", fixed_fp16_path)

In [ ]:
import time
import tensorrt as trt

fp16_config = fp16_builder.create_builder_config()

fp16_config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    2 << 30
)

fp16_config.profiling_verbosity = (
    trt.ProfilingVerbosity.DETAILED
)

# Dynamic optimization profile
fp16_profile = fp16_builder.create_optimization_profile()

fp16_profile.set_shape(
    "pixel_values",
    min=(1, 3, 224, 224),
    opt=(8, 3, 224, 224),
    max=(32, 3, 224, 224)
)

# TensorRT 11.4: validate the profile itself.
assert bool(fp16_profile), "Invalid optimization profile"

print("Optimization profile:")
print(fp16_profile.get_shape("pixel_values"))

fp16_config.add_optimization_profile(fp16_profile)

# Save directly to Google Drive.
fp16_engine_path = (
    output_dir / "clip_baseline_fp16.engine"
)

print("\nBuilding FP16 TensorRT baseline...")

start = time.perf_counter()

serialized = fp16_builder.build_serialized_network(
    fp16_network,
    fp16_config
)

if serialized is None:
    raise RuntimeError("FP16 engine build failed")

fp16_engine_path.write_bytes(serialized)

fp16_runtime = trt.Runtime(fp16_logger)

fp16_engine = fp16_runtime.deserialize_cuda_engine(
    fp16_engine_path.read_bytes()
)

assert fp16_engine is not None

print("\nFP16 baseline built successfully!")
print(f"Build time: {time.perf_counter() - start:.2f} s")
print(
    f"Engine size: "
    f"{fp16_engine_path.stat().st_size / 1024**2:.2f} MiB"
)
print("Saved to:", fp16_engine_path)

print("\nEngine I/O:")
for i in range(fp16_engine.num_io_tensors):
    name = fp16_engine.get_tensor_name(i)
    print(
        name,
        fp16_engine.get_tensor_dtype(name),
        fp16_engine.get_tensor_shape(name)
    )

## 11. Benchmark the two TensorRT engines on the same GPU

CUDA events measure **vision encoder GPU execution**, not JPEG decoding or end-to-end latency. The C++ application has separate end-to-end measurements.

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import torch
import tensorrt as trt

def benchmark_trt_engine(
    engine,
    batch_size,
    warmup=30,
    iterations=100
):
    context = engine.create_execution_context()

    assert context is not None

    input_name = "pixel_values"
    output_name = "image_features"

    dtype_map = {
        trt.DataType.FLOAT: torch.float32,
        trt.DataType.HALF: torch.float16
    }

    input_dtype = dtype_map[
        engine.get_tensor_dtype(input_name)
    ]

    output_dtype = dtype_map[
        engine.get_tensor_dtype(output_name)
    ]

    input_shape = (batch_size, 3, 224, 224)

    assert context.set_input_shape(
        input_name,
        input_shape
    )

    output_shape = tuple(
        context.get_tensor_shape(output_name)
    )

    images = torch.randn(
        input_shape,
        dtype=input_dtype,
        device="cuda"
    ).contiguous()

    features = torch.empty(
        output_shape,
        dtype=output_dtype,
        device="cuda"
    )

    context.set_tensor_address(
        input_name,
        images.data_ptr()
    )

    context.set_tensor_address(
        output_name,
        features.data_ptr()
    )

    stream = torch.cuda.current_stream()

    # Ensure allocations and initialization complete.
    stream.synchronize()

    # Warm up TensorRT and CUDA kernels.
    for _ in range(warmup):
        ok = context.execute_async_v3(
            stream_handle=stream.cuda_stream
        )

        if not ok:
            raise RuntimeError("TensorRT warmup failed")

    stream.synchronize()

    latencies = []

    for _ in range(iterations):
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record(stream)

        ok = context.execute_async_v3(
            stream_handle=stream.cuda_stream
        )

        if not ok:
            raise RuntimeError("TensorRT inference failed")

        end_event.record(stream)
        end_event.synchronize()

        latencies.append(
            start_event.elapsed_time(end_event)
        )

    latencies = np.asarray(latencies)

    return {
        "batch_size": batch_size,
        "median_ms": float(np.median(latencies)),
        "mean_ms": float(np.mean(latencies)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "throughput_img_s": float(
            batch_size * 1000.0 / np.mean(latencies)
        )
    }

In [ ]:
import pandas as pd

# Benchmarks execute sequentially on the same GPU.
# Warm-up and measurement settings are identical.

comparison_results = []

engines = [
    ("FP16 TensorRT", fp16_engine),
    ("Mixed INT8/FP16 TensorRT", mixed_engine)
]

for name, engine in engines:
    for batch_size in [1, 32]:

        print(f"Benchmarking {name}, batch={batch_size}")

        result = benchmark_trt_engine(
            engine,
            batch_size=batch_size,
            warmup=50,
            iterations=150
        )

        result["model"] = name
        comparison_results.append(result)

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df[
    [
        "model",
        "batch_size",
        "median_ms",
        "mean_ms",
        "p95_ms",
        "p99_ms",
        "throughput_img_s"
    ]
]

print("\nFinal same-session TensorRT comparison:")
display(comparison_df)

comparison_df.to_csv(
    results_dir / "final_trt_comparison.csv",
    index=False
)

print("\nSpeed comparison:")

for batch_size in [1, 32]:
    baseline = comparison_df[
        (comparison_df["model"] == "FP16 TensorRT")
        & (comparison_df["batch_size"] == batch_size)
    ].iloc[0]

    mixed = comparison_df[
        (comparison_df["model"] == "Mixed INT8/FP16 TensorRT")
        & (comparison_df["batch_size"] == batch_size)
    ].iloc[0]

    speedup = baseline["mean_ms"] / mixed["mean_ms"]

    print(
        f"Batch {batch_size}: "
        f"{speedup:.3f}x relative to FP16"
    )

print("\nResults saved to Google Drive.")

## 12. Validate the FP16 accuracy baseline

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

fp16_context = fp16_engine.create_execution_context()

assert fp16_context is not None

fp16_predictions = []
fp16_embeddings = []

stream = torch.cuda.current_stream()

normalized_text = F.normalize(
    text_features.float().to(device),
    dim=-1
)

with torch.inference_mode():

    for images in evaluation_loader:

        images = images.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True
        ).contiguous()

        assert fp16_context.set_input_shape(
            "pixel_values",
            tuple(images.shape)
        )

        output_shape = tuple(
            fp16_context.get_tensor_shape(
                "image_features"
            )
        )

        output = torch.empty(
            output_shape,
            dtype=torch.float32,
            device=device
        )

        fp16_context.set_tensor_address(
            "pixel_values",
            images.data_ptr()
        )

        fp16_context.set_tensor_address(
            "image_features",
            output.data_ptr()
        )

        ok = fp16_context.execute_async_v3(
            stream_handle=stream.cuda_stream
        )

        if not ok:
            raise RuntimeError("FP16 TensorRT inference failed")

        features = F.normalize(
            output.float(),
            dim=-1
        )

        scores = features @ normalized_text.T

        fp16_predictions.extend(
            scores.argmax(dim=-1).cpu().numpy()
        )

        fp16_embeddings.append(features.cpu())

stream.synchronize()

fp16_predictions = np.asarray(
    fp16_predictions,
    dtype=np.int64
)

fp16_embeddings = torch.cat(
    fp16_embeddings,
    dim=0
)

fp16_accuracy = accuracy_score(
    y_true,
    fp16_predictions
)

fp16_macro_f1 = f1_score(
    y_true,
    fp16_predictions,
    labels=[0, 1, 2],
    average="macro"
)

fp16_agreement = np.mean(
    fp16_predictions == fp32_predictions
)

print("FP16 TensorRT accuracy:", f"{fp16_accuracy:.4f}")
print("FP16 macro F1:", f"{fp16_macro_f1:.4f}")
print("FP32 prediction agreement:", f"{fp16_agreement:.4f}")

print("\nFP16 confusion matrix:")
print(
    confusion_matrix(
        y_true,
        fp16_predictions,
        labels=[0, 1, 2]
    )
)

print("\nAccuracy comparison:")
print(f"FP32 PyTorch:       {fp32_accuracy:.4f}")
print(f"FP16 TensorRT:      {fp16_accuracy:.4f}")
print(f"Mixed INT8/FP16:    {mixed_accuracy:.4f}")

np.save(
    results_dir / "fp16_trt_predictions.npy",
    fp16_predictions
)

print("\nFP16 validation predictions saved to Google Drive.")

## 13. Measured outcome (Tesla T4, 300-image balanced evaluation)

| Metric | FP32 PyTorch | FP16 TensorRT | Selective INT8/FP16 TensorRT |
|---|---:|---:|---:|
| Accuracy | 81.00% | 80.67% | 81.33% |
| Macro F1 | 0.7991 | 0.7951 | 0.8030 |
| Engine size | - | 166.14 MiB | 114.42 MiB |
| Batch 1 mean GPU latency | - | 4.410 ms | 4.229 ms |
| Batch 32 mean GPU latency | - | 91.727 ms | 84.006 ms |
| Batch 32 throughput | - | 348.9 images/s | 380.9 images/s |

**C++ deployment:** see `main.cpp` and `build.sh`. Final standalone C++ accuracy: **80.67%**; 300-image local-storage mean end-to-end latency: **33.77 ms** (GPU initialization excluded). Measurements from the original Colab runs; fresh runs can differ. Dawn/dusk recall is relatively low. Text embeddings are generated in Section 2 and copied into the Git release as `text_embeddings_fp32.bin`.

**Reproducibility and limitations:** pretrained frozen CLIP, no new training, this is a small balanced evaluation subset, not validation of all BDD100K conditions. TensorRT engines may be incompatible with different GPUs, CUDA drivers or TensorRT versions. Follow the dataset and model distribution terms.